# Module 5: Wednesday Session
## CS 5045: Computation for the Data Sciences

**Today's plan:** live demo of every function from the module lecture, applied to the road race dataset.

We'll label each function with the computational operation it packages:
Represent · Filter · Transform · Aggregate · Communicate

**New this module:** File I/O: what Python does when it opens a file, and how that connects to `pd.read_csv()`.

---
## Setup: REPRESENT

Loading the dataset into memory is the **Represent** operation.
Once it's in a DataFrame, every other operation can act on it.

In [32]:
import pandas as pd

df = pd.read_csv('data.csv')
print(df.shape)   # how many rows and columns did we load?
df.head()

(16, 6)


,runner_id,name,finish_time_min,age,gender,category
0,R001,Amara Osei,42.3,29,F,Open
1,R002,Brian Tran,38.7,34,M,Open
2,R003,Celia Moreno,51.2,47,F,Masters
3,R004,David Park,44.8,31,M,Open
4,R005,Fatima Al-Hassan,56.4,52,F,Masters


---
## Transform: pace_per_km

**Operation:** Transform: takes a finish time and produces a new value (pace).

**Design habit:** write the docstring *before* the body.
If you can't describe what the function does in one sentence, the function is probably doing too many things.


In [35]:
def pace_per_km(finish_time_min, distance_km=10.0):
    """
    TRANSFORM: compute pace (min/km) from total finish time and race distance.

    Parameters:
        finish_time_min (float) -- total finish time in minutes
        distance_km     (float) -- race distance in km (default 10.0)

    Returns:
        float -- pace in minutes per kilometer, rounded to 2 decimal places
    """
    return round(finish_time_min / distance_km, 2)

# Test on a single value before applying to the whole column
print('Idris Okafor pace (10K):', pace_per_km(35.6))
print('Same time, 5K race:    ', pace_per_km(35.6, distance_km=5.0))

Idris Okafor pace (10K): 3.56
Same time, 5K race:     7.12


In [36]:
# Apply to the entire column -- appl --> 16 function calls
df['pace_min_per_km'] = df['finish_time_min'].apply(pace_per_km)
df[['name', 'finish_time_min', 'pace_min_per_km']].head(8)

,name,finish_time_min,pace_min_per_km
0,Amara Osei,42.3,4.23
1,Brian Tran,38.7,3.87
2,Celia Moreno,51.2,5.12
3,David Park,44.8,4.48
4,Fatima Al-Hassan,56.4,5.64
5,George Kim,40.1,4.01
6,Hannah Schmidt,48.9,4.89
7,Idris Okafor,35.6,3.56


---
## Transform: age_group

**Operation:** Transform: maps an integer age to a category label.

No rows are removed; a new value is computed for every row. That's what distinguishes Transform from Filter.


In [37]:
def age_group(age):
    """
    TRANSFORM: classify a runner into an age group string.

    Parameters:
        age (int) -- runner's age in years

    Returns:
        str -- 'Under 30', '30-39', '40-49', or '50+'
    """
    if age < 30:
        return 'Under 30'
    elif age < 40:
        return '30-39'
    elif age < 50:
        return '40-49'
    else:
        return '50+'

df['age_group'] = df['age'].apply(age_group)
df[['name', 'age', 'age_group']].head(8)

,name,age,age_group
0,Amara Osei,29,Under 30
1,Brian Tran,34,30-39
2,Celia Moreno,47,40-49
3,David Park,31,30-39
4,Fatima Al-Hassan,52,50+
5,George Kim,26,Under 30
6,Hannah Schmidt,44,40-49
7,Idris Okafor,22,Under 30


---
## Filter: is_podium

**Operation:** Filter: asks a yes/no question about a single row.

`is_podium` returns a boolean. That boolean is the *predicate* for a filter operation.
Two steps: (1) create the boolean column with `.apply(is_podium)`, (2) filter with `df[df['is_podium']]`.



In [38]:
def is_podium(finish_time_min, cutoff=41.0):
    """
    FILTER helper: return True if the finish time qualifies for a podium spot.

    Parameters:
        finish_time_min (float) -- runner's finish time in minutes
        cutoff          (float) -- podium cutoff time in minutes (default 41.0)

    Returns:
        bool -- True if finish_time_min <= cutoff
    """
    return finish_time_min <= cutoff

df['is_podium'] = df['finish_time_min'].apply(is_podium)

# Now use the boolean column to filter
podium_df = df[df['is_podium']]
print(f'{len(podium_df)} podium finishers (cutoff: 41 min):')
podium_df[['name', 'finish_time_min', 'age_group', 'category']]

4 podium finishers (cutoff: 41 min):


,name,finish_time_min,age_group,category
1,Brian Tran,38.7,30-39,Open
5,George Kim,40.1,Under 30,Open
7,Idris Okafor,35.6,Under 30,Open
11,Marcus Webb,37.2,Under 30,Open


---
## Aggregate: summarize_category

**Operation:** Aggregate: collapses many rows into one summary per category.

The function filters to one category, then computes mean, min, and max. Call it once per category and compare.

In [39]:
def summarize_category(dataframe, category_name):
    """
    AGGREGATE: compute mean, min, and max finish time for one category.

    Parameters:
        dataframe     (DataFrame) -- the full race results DataFrame
        category_name (str)       -- the category to filter for ("Open" or "Masters")

    Returns:
        dict -- {"mean": float, "min": float, "max": float}
    """
    group = dataframe[dataframe["category"] == category_name]
    return {
        "mean": float(round(group["finish_time_min"].mean(), 2)),
        "min":  float(group["finish_time_min"].min()),
        "max":  float(group["finish_time_min"].max()),
    }

masters_summary = summarize_category(df, "Masters")
open_summary    = summarize_category(df, "Open")
print(f'Rows in: {len(df)}  |  Each call returns 1 dict for one category.')
print('Masters:', masters_summary)
print('Open:   ', open_summary)

Rows in: 16  |  Each call returns 1 dict for one category.
Masters: {'mean': 54.0, 'min': 48.9, 'max': 61.0}
Open:    {'mean': 41.95, 'min': 35.6, 'max': 49.4}


---
## Communicate: print_leaderboard

**Operation:** Communicate: presents a result to a human reader.

Notice the default `top_n=5`. Call it twice with different values to show how defaults work.



In [43]:
def print_leaderboard(data, top_n=5):
    """
    COMMUNICATE: print the top-N runners sorted by finish time.

    Parameters:
        data   (pd.DataFrame) -- race results
        top_n  (int)          -- number of runners to display (default 5)
    """
    leaders = (
        data.nsmallest(top_n, 'finish_time_min')
            [['name', 'finish_time_min', 'pace_min_per_km', 'age_group', 'category']]
    )
    print(f'=== Top {top_n} Finishers ===')
    print(leaders.to_string(index=False))
    print()

# Default: top 5
print_leaderboard(df)

# Override: top 3
print_leaderboard(df, top_n=3)

=== Top 5 Finishers ===
        name  finish_time_min  pace_min_per_km age_group category
Idris Okafor             35.6             3.56  Under 30     Open
 Marcus Webb             37.2             3.72  Under 30     Open
  Brian Tran             38.7             3.87     30-39     Open
  George Kim             40.1             4.01  Under 30     Open
  Raj Kapoor             41.6             4.16     30-39     Open

=== Top 3 Finishers ===
        name  finish_time_min  pace_min_per_km age_group category
Idris Okafor             35.6             3.56  Under 30     Open
 Marcus Webb             37.2             3.72  Under 30     Open
  Brian Tran             38.7             3.87     30-39     Open



---
## Variable Scope

A variable defined inside a function does not exist outside it.
Only the `return` value crosses the boundary.



In [ ]:
def demo_scope():
    local_var = 'I only exist inside demo_scope'
    return local_var

result = demo_scope()
print('Return value:', result)   # works -- we captured the return value

In [ ]:
# Uncomment the next line to see the NameError:
# print(local_var)

---
## File I/O: What Is Under the Hood

Every time you call `pd.read_csv()`, Python opens a file, reads its lines, and parses them.
Here's what that looks like one level down.

**Why this matters:**
- You will encounter file formats that pandas doesn't handle directly (log files, custom exports, plain text).
- Understanding `open()` makes `read_csv()` less magic and more tool.
- Writing a `read_file()` function is practice for the same pattern you've been applying all module: wrap a repeated operation into a named function.



In [44]:
# Step 1: Write a small text file so we have something to read.
# 'w' mode: create the file (or overwrite it if it already exists).

lines_to_write = [
    "runner_id,name,finish_time_min\n",
    "R001,Idris Okafor,35.6\n",
    "R002,Priya Nair,37.2\n",
    "R003,Sam Torres,39.4\n",
]

with open("temp_race_data.txt", "w") as f:
    f.writelines(lines_to_write)

print("File written: temp_race_data.txt")
print(f"Wrote {len(lines_to_write)} lines.")

File written: temp_race_data.txt
Wrote 4 lines.


In [45]:
# Step 2: Read the file back with open() in 'r' (read) mode.
# readlines() returns a list -- one string per line, including the \n.

with open("temp_race_data.txt", "r") as f:
    raw_lines = f.readlines()

print(f"Read {len(raw_lines)} lines:")
print(raw_lines)   # show the raw strings -- notice the \n at the end of each

print()
print("After stripping whitespace:")
for line in raw_lines:
    print(line.strip())

Read 4 lines:
['runner_id,name,finish_time_min\n', 'R001,Idris Okafor,35.6\n', 'R002,Priya Nair,37.2\n', 'R003,Sam Torres,39.4\n']

After stripping whitespace:
runner_id,name,finish_time_min
R001,Idris Okafor,35.6
R002,Priya Nair,37.2
R003,Sam Torres,39.4


In [46]:
# Step 3: Wrap the open/read/strip logic in a function.
# Operation: REPRESENT -- turning a file on disk into a Python object in memory.

def read_file(path):
    """
    REPRESENT: read a text file and return its lines as a list of strings.

    Parameters:
        path (str) -- path to the file to open

    Returns:
        list of str -- one string per line, with leading/trailing whitespace removed
    """
    with open(path, "r") as f:
        lines = f.readlines()
    return [line.strip() for line in lines]


result = read_file("temp_race_data.txt")
print(f"read_file() returned {len(result)} items:")
for row in result:
    print(row)

read_file() returned 4 items:
runner_id,name,finish_time_min
R001,Idris Okafor,35.6
R002,Priya Nair,37.2
R003,Sam Torres,39.4


In [47]:
# Step 4: Run pd.read_csv() on the same file and compare.
# pandas calls open() for you, then also: splits on commas, uses row 0 as the header,
# and infers that finish_time_min should be float64, not a string.

df_from_file = pd.read_csv("temp_race_data.txt")
print("pandas read the same file:")
print(df_from_file)
print()
print("Column types:")
print(df_from_file.dtypes)

print()
print("Your read_file() gave you raw strings.")
print("pandas gave you a typed DataFrame.")
print("read_csv() = open() + parse + infer types. Now you know what is underneath.")

# Clean up the temp file.
import os #os interacts with computer operating system
os.remove("temp_race_data.txt")
print("\nTemp file removed.")

pandas read the same file:
  runner_id          name  finish_time_min
0      R001  Idris Okafor             35.6
1      R002    Priya Nair             37.2
2      R003    Sam Torres             39.4

Column types:
runner_id              str
name                   str
finish_time_min    float64
dtype: object

Your read_file() gave you raw strings.
pandas gave you a typed DataFrame.
read_csv() = open() + parse + infer types. Now you know what is underneath.

Temp file removed.


---
## Connecting to the Assignment

The assignment asks you to write functions using the road race dataset, one per operation:

| Operation | What to write |
|-----------|---------------|
| **Transform** | A function that computes a new value from one or more columns |
| **Filter** | A function that returns True/False for a row-level condition |
| **Aggregate** | A function that takes a DataFrame and a category name, and returns a summary dict |
| **Communicate** | A function that prints or displays results for a human reader |

**Starter pattern** (copy this and fill in your own details):

```python
def my_transform(value, parameter=default):
    """
    TRANSFORM: describe what this does in one sentence.
    Parameters: value (type) -- what it represents
    Returns: type -- what comes back
    """
    return ...  # your calculation here
```

**Also try before you submit:** Create `read_file.py` in VS Code, paste the `read_file()` function from this notebook, and run it from the terminal:
```
python read_file.py
```
Point it at a plain text file on your machine. If it prints lines, your environment is working and you understand what `open()` does.

Questions for yourself before you submit:
- Does every function have a docstring?
- Does every function have a `return` statement (except Communicate)?
- Have you tested each function on a single value before calling `.apply()`?
- Can you label each function with the correct operation name?